# WGQ Model — Fixed Implementation (Notebook 09)

**Fixes vs Notebook 08:**
1. Passage text loaded from  — Stream 1 CLS now sees the actual passage
2. EyeBench pre-built fold CSVs used directly — splits match baselines exactly
3. Normalizers fit on training data only — no leakage
4.  join key — matches EyeBench 9,718 trials exactly

**After any Colab runtime restart: run Cell 1 (installs), then Cell 2 (setup + load all data), then skip to whichever section you need.**

In [ ]:
!pip install -q transformers torch torchvision scikit-learn pandas numpy tqdm pyarrow

In [ ]:
# ── Imports and drive mount ───────────────────────────────────────────────────
import os, sys, json, math, importlib, warnings
import numpy as np
import pandas as pd
import torch
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

PROJECT_CODE = "/content/drive/MyDrive/eyebench_project/Project_codebase"
if PROJECT_CODE not in sys.path:
    sys.path.insert(0, PROJECT_CODE)

import config, data_loader, folds, tokenizer_utils
import model as model_lib
import dataset as ds_lib
import trainer
for mod in [config, data_loader, folds, tokenizer_utils, model_lib, ds_lib, trainer]:
    importlib.reload(mod)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

for d in [config.CACHE_DIR, config.RESULTS_DIR, config.CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark        = True
try:   torch.backends.cuda.enable_flash_sdp(True)
except Exception: pass

# ── Load all data (re-run this cell after any runtime restart) ────────────────
print()
print("=" * 55)
print("STEP 1/3  Loading trials_df ...")
print("=" * 55)
trials_df = data_loader.build_trials_df(
    ia_path         = config.IA_PATH,
    paragraphs_path = config.PARAGRAPHS_PATH,
    cache_path      = config.TRIALS_CACHE_PATH,
)
passage_ok   = (trials_df[config.PASSAGE_COL].str.len() > 0).mean()
question_ok  = (trials_df[config.QUESTION_COL].str.len() > 0).mean() if config.QUESTION_COL in trials_df.columns else 0
print(f"  trials_df: {len(trials_df):,} rows")
print(f"  Passage coverage : {passage_ok:.1%}  (should be >95%)")
print(f"  Question coverage: {question_ok:.1%}  (should be >95%)")

print()
print("=" * 55)
print("STEP 2/3  Loading word-gaze sequences ...")
print("=" * 55)
word_gaze = data_loader.build_word_gaze_cache(
    ia_path    = config.IA_PATH,
    cache_path = config.WORD_GAZE_CACHE_PATH,
)
wg_coverage = sum(1 for tid in trials_df[config.UNIQUE_TRIAL_COL] if tid in word_gaze) / len(trials_df)
print(f"  word_gaze: {len(word_gaze):,} sequences  |  coverage in trials_df: {wg_coverage:.1%}")

print()
print("=" * 55)
print("STEP 3/3  Pre-tokenizing passage + question ...")
print("=" * 55)
from transformers import RobertaTokenizerFast
tokenizer = RobertaTokenizerFast.from_pretrained(config.ROBERTA_NAME)
tokenized = tokenizer_utils.build_tokenized_tensors(
    trials_df  = trials_df,
    tokenizer  = tokenizer,
    cache_path = config.TOKENIZED_CACHE_PATH,
)
has_passage_tokens = (tokenized["passage_wids"] >= 0).any(dim=1).float().mean().item()
print(f"  Trials with passage tokens in joint encoding: {has_passage_tokens:.1%}  (should be >95%)")

print()
print("All data loaded. Ready to train.")

---
## Section 1: Verify EyeBench Fold Splits

Pre-built  files from the EyeBench repo.
Check that our  covers all 9,718 EyeBench trial IDs.

In [ ]:
print("EyeBench fold split sizes:")
folds.print_fold_table(trials_df, fold_meta_dir=config.FOLD_META_DIR)

# Coverage check across all 10 folds
all_eb_ids = set()
for k in range(config.N_FOLDS):
    fold_csv = _pd.read_csv(f"{config.FOLD_META_DIR}/fold_{k}_trial_ids_by_regime.csv")
    all_eb_ids.update(fold_csv[config.UNIQUE_TRIAL_COL].values)

our_ids = set(trials_df[config.UNIQUE_TRIAL_COL].values)
print(f"
Total unique EyeBench trial IDs (all folds): {len(all_eb_ids):,}")
print(f"Found in trials_df: {len(all_eb_ids & our_ids):,}  ({len(all_eb_ids & our_ids)/len(all_eb_ids):.1%})")
print(f"Missing:            {len(all_eb_ids - our_ids):,}")

---
## Section 2: 10-Fold Cross-Validation

One model per fold, evaluated simultaneously on all 3 generalization regimes.
Results saved after every fold — safe to interrupt and resume.

In [ ]:
results_file = f"{config.RESULTS_DIR}/fold_results.json"
fold_results = {}

if os.path.exists(results_file):
    with open(results_file) as f:
        fold_results = json.load(f)
    print(f"Resuming: folds {sorted(fold_results.keys())} already done.")

for fold_k in range(config.N_FOLDS):
    fold_key = str(fold_k)
    if fold_key in fold_results:
        print(f"Fold {fold_k}: done — skipping.")
        continue

    print(f"
{'='*54}")
    print(f"FOLD {fold_k} / {config.N_FOLDS - 1}")
    print(f"{'='*54}")

    # EyeBench pre-built splits (Fix 2)
    train_df, val_df, r1_df, r2_df, r3_df = folds.load_fold_splits(
        fold_k, trials_df, config.FOLD_META_DIR
    )
    if min(len(r1_df), len(r2_df), len(r3_df)) == 0:
        print(f"  Fold {fold_k}: empty test split — skipping.")
        continue

    # DataLoaders — normalizers fit on train only (Fix 3)
    train_loader, val_loader, test_loaders = ds_lib.make_loaders(
        train_df, val_df, [r1_df, r2_df, r3_df],
        tokenized, word_gaze, trials_df,
    )

    # Build and compile model
    wgq_model = model_lib.WGQModel(
        num_ia_features=config.NUM_IA_FEATURES,
        num_gaze_stats=config.NUM_GAZE_STATS,
    ).to(DEVICE)
    print(f"  Trainable params: {sum(p.numel() for p in wgq_model.parameters() if p.requires_grad):,}")
    try:
        wgq_model = torch.compile(wgq_model, dynamic=True)
        print("  torch.compile: enabled")
    except Exception as e:
        print(f"  torch.compile unavailable ({e})")

    # Train with Colab-resume checkpointing
    best = trainer.train_fold(
        wgq_model, train_loader, val_loader,
        fold_k=fold_k, variant_name="wgq", device=DEVICE,
    )
    if best is None:
        print(f"  Fold {fold_k}: training failed (no improvement) — skipping.")
        continue

    # Threshold optimised on VAL set, applied to TEST set
    threshold = trainer.optimize_threshold(best["val_logits"], best["val_labels"])
    print(f"  Val-optimal threshold: {threshold:.2f}")

    fold_results[fold_key] = {"threshold": threshold}
    for regime_name, test_loader in zip(config.REGIME_NAMES, test_loaders):
        logits_np, labels_np = trainer.collect_preds(wgq_model, test_loader, DEVICE)
        auroc, bal_acc = trainer.evaluate(logits_np, labels_np, threshold=threshold)
        fold_results[fold_key][regime_name] = {
            "auroc":   round(auroc,   2),
            "bal_acc": round(bal_acc, 2),
            "n":       len(test_loader.dataset),
        }
        print(f"  {regime_name}: AUROC={auroc:.1f}  BalAcc={bal_acc:.1f}")

    with open(results_file, "w") as f:
        json.dump(fold_results, f, indent=2)
    print(f"  Fold {fold_k} saved → {results_file}")

    del wgq_model, train_loader, val_loader, test_loaders
    torch.cuda.empty_cache()

print("
=== All folds complete ===")

---
## Section 3: Aggregate Results

In [ ]:
# Load results from disk in case this cell is run after a restart
if not fold_results:
    if os.path.exists(results_file):
        with open(results_file) as f:
            fold_results = json.load(f)
    else:
        raise RuntimeError(f"No results found at {results_file}. Run Section 2 first.")

summary = trainer.aggregate_fold_results(fold_results)

print("=== WGQModel: Mean ± SEM across folds ===")
trainer.print_results_table(summary)

print("
=== Comparison to EyeBench baselines ===")
trainer.print_comparison_table(summary)

with open(f"{config.RESULTS_DIR}/aggregate_results.json", "w") as f:
    json.dump(summary, f, indent=2)
print("
Saved aggregate_results.json")

---
## Section 4: Ablation Study (Fold 0, all 3 regimes)

| Variant | Change | Tests |
|---|---|---|
| Full WGQModel | — | Baseline |
| A1: No Q-conditioning | Self-attn instead of cross-attn | Does question gating help? |
| A2: Text-only | Remove all gaze | Does gaze add signal? |
| A3: No global stats | Zero stat vector | Do handcrafted features matter? |
| A4: No word-level IA | Zero word_gaze tensor | Is word-level IA needed beyond text? |

In [ ]:
ABLATION_FOLD = 0

train_abl, val_abl, r1_abl, r2_abl, r3_abl = folds.load_fold_splits(
    ABLATION_FOLD, trials_df, config.FOLD_META_DIR
)
train_ld_abl, val_ld_abl, test_lds_abl = ds_lib.make_loaders(
    train_abl, val_abl, [r1_abl, r2_abl, r3_abl],
    tokenized, word_gaze, trials_df,
)

abl_file = f"{config.RESULTS_DIR}/ablation_results.json"
abl_results = {}
if os.path.exists(abl_file):
    with open(abl_file) as f:
        abl_results = json.load(f)
    print(f"Resuming ablations: {list(abl_results.keys())} done.")

for abl_key, abl_label in config.ABLATION_VARIANTS:
    if abl_key in abl_results:
        print(f"Skip: {abl_label}")
        continue

    print(f"
=== {abl_label} ===")
    abl_model = model_lib.build_model(
        abl_key,
        num_ia_features=config.NUM_IA_FEATURES,
        num_gaze_stats=config.NUM_GAZE_STATS,
    ).to(DEVICE)
    print(f"  Trainable: {sum(p.numel() for p in abl_model.parameters() if p.requires_grad):,}")
    try:
        abl_model = torch.compile(abl_model, dynamic=True)
    except Exception:
        pass

    best = trainer.train_fold(
        abl_model, train_ld_abl, val_ld_abl,
        fold_k=ABLATION_FOLD, variant_name=f"abl_{abl_key}", device=DEVICE,
    )
    if best is None:
        print(f"  {abl_label}: failed — skipping.")
        continue

    thresh = trainer.optimize_threshold(best["val_logits"], best["val_labels"])
    abl_results[abl_key] = {"label": abl_label, "threshold": thresh, "regimes": {}}

    for rname, tl in zip(config.REGIME_NAMES, test_lds_abl):
        logits_np, labels_np = trainer.collect_preds(abl_model, tl, DEVICE)
        auroc, ba = trainer.evaluate(logits_np, labels_np, threshold=thresh)
        abl_results[abl_key]["regimes"][rname] = {"auroc": round(auroc,1), "bal_acc": round(ba,1)}
        print(f"  {rname}: AUROC={auroc:.1f}  BalAcc={ba:.1f}")

    with open(abl_file, "w") as f:
        json.dump(abl_results, f, indent=2)
    del abl_model
    torch.cuda.empty_cache()

print("
=== All ablations complete ===")

In [ ]:
# Load from disk if needed
if not abl_results and os.path.exists(abl_file):
    with open(abl_file) as f:
        abl_results = json.load(f)

if not abl_results:
    print("No ablation results found. Run Section 4 first.")
else:
    print("=== Ablation Results (fold 0, all 3 regimes) ===")
    hdr = f"{'Variant':<46}" + "".join(f" {r+' AUROC':>12} {r+' BalAcc':>12}" for r in config.REGIME_SHORT)
    print(hdr)
    print("-" * (46 + 3*26))

    full_aurocs = None
    for abl_key, abl_label in config.ABLATION_VARIANTS:
        if abl_key not in abl_results:
            continue
        line, aurocs = f"{abl_label:<46}", []
        for rn in config.REGIME_NAMES:
            m = abl_results[abl_key]["regimes"].get(rn, {})
            a, b = m.get("auroc", float("nan")), m.get("bal_acc", float("nan"))
            line += f" {a:>12.1f} {b:>12.1f}"
            aurocs.append(a)
        print(line)
        if abl_key == "full_wgq":
            full_aurocs = aurocs

    if full_aurocs:
        print("
ΔAUROC vs Full WGQModel:")
        for abl_key, abl_label in config.ABLATION_VARIANTS:
            if abl_key == "full_wgq" or abl_key not in abl_results:
                continue
            deltas = [
                abl_results[abl_key]["regimes"].get(rn, {}).get("auroc", float("nan")) - full_aurocs[i]
                for i, rn in enumerate(config.REGIME_NAMES)
            ]
            print(f"  {abl_label:<44}: {'  '.join(f'{d:+.1f}' for d in deltas)}")

---
## Section 5: Final Summary

In [ ]:
# Reload summary if needed
if "summary" not in dir():
    agg_path = f"{config.RESULTS_DIR}/aggregate_results.json"
    if os.path.exists(agg_path):
        with open(agg_path) as f:
            summary = json.load(f)
    else:
        raise RuntimeError("Run Section 3 first.")

print("=" * 70)
print("FINAL RESULTS: WGQModel (Word-Gaze-Question) — Fixed Implementation")
print("=" * 70)
trainer.print_results_table(summary)
trainer.print_comparison_table(summary)
print()
print("Architecture:")
print("  Stream 1: frozen RoBERTa-base([CLS] passage [SEP][SEP] Question [SEP]) → CLS (768-d)")
print("  Stream 2: per-word (RoBERTa emb + 12 IA gaze features) → cross-attn(question) → pool (256-d)")
print("  Stream 3: 6 global gaze statistics")
print("  Classifier: MLP(768 + 256 + 6) → logit")
print()
print("Fixes applied vs Notebook 08:")
print("  ✓ Passage text loaded from trial_level_paragraphs.csv")
print("  ✓ EyeBench pre-built fold CSVs used (exact splits)")
print("  ✓ Normalizers fit on training data only per fold")
print("  ✓ unique_trial_id join key → matches EyeBench 9,718 trials")